In [1]:
# Test script 2

In [1]:
# Test calculating pop weighted AF for ozone

In [1]:
import os
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [2]:
# === Path config ===
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
O3_DIR = "/glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/"

In [3]:
# === Load data ===
pop_file = "ssp2_coarse_grid_annual_2000-2100.nc"
pop_path = os.path.join(POP_DIR, pop_file)
population = xr.open_dataarray(pop_path)

In [4]:
# === Path config ===
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

in_file = "GBD_Country_Masks_0.10.nc"
in_path = os.path.join(MASK_DIR, in_file)
country_mask = xr.open_dataarray(in_path)

In [5]:
o3_file = "OSDMA8_BC_CESM2_ARISE_01_2035-2068.nc"
o3_path = os.path.join(O3_DIR, o3_file)
o3 = xr.open_dataarray(o3_path)

# Adjust indices to match (with small tolerance)
# e.g., max 1e-7 km distance
o3 = o3.reindex_like(country_mask, method="nearest", tolerance=1e-9, fill_value=0)
population = population.reindex_like(country_mask, method="nearest", tolerance=1e-9)

In [6]:
n_samples = 1000

# === Scalar distributions (assuming normal dist.) ===
# TMREL from GBD21
tmrel_mean = 32.4
tmrel_std = (35.7 - 29.1) / (2 * 1.96)
tmrel_samples = np.random.normal(tmrel_mean, tmrel_std, size=n_samples)

# Wrap in xarray for nice labeling:
tmrel_samples = xr.DataArray(
    tmrel_samples,
    dims=("sample"),
    coords={"sample": np.arange(n_samples)}
)

# Beta from RR per 10ppb
RR_10 = 1.074
RR_10_lower = 1.014
RR_10_upper = 1.137
beta_mean = np.log(RR_10) / 10
beta_std = (np.log(RR_10_upper) - np.log(RR_10_lower)) / (2 * 1.96 * 10)
beta_samples = np.random.normal(beta_mean, beta_std, size=n_samples)

# Wrap in xarray for nice labeling:
beta_samples = xr.DataArray(
    beta_samples,
    dims=("sample"),
    coords={"sample": np.arange(n_samples)}
)

In [7]:
O3_diff = o3.chunk({"lat": 18, "lon": 36}) - tmrel_samples

In [8]:
TMREL_O3 = xr.where(O3_diff > 0, O3_diff, 0)  # where the difference < 0 set to 0

In [9]:
RR = np.exp(beta_samples * TMREL_O3)

In [13]:
AF = (RR-1)/RR

In [10]:
pop_years = population.sel(year=RR.year).chunk({"lat": 18, "lon": 36})

In [14]:
weighted_value = pop_years * AF

In [15]:
country_list = []

for i in range(204):
    mask = country_mask.isel(country=i)
    country_weight = xr.where(mask == 1, weighted_value, np.nan).sum(dim=("lat", "lon"))
    pop_country = xr.where(mask == 1, pop_years, np.nan).sum(dim=("lat", "lon"))
    country_pop_weighted = country_weight / pop_country
    country_list.append(country_pop_weighted)

In [16]:
PAF = xr.concat(country_list, "country")

In [19]:
paf_save = PAF.load()


KeyboardInterrupt



In [20]:
PAF

<xarray.DataArray (country: 204, year: 34, sample: 1000)> Size: 55MB
dask.array<concatenate, shape=(204, 34, 1000), dtype=float64, chunksize=(1, 34, 1000), chunktype=numpy.ndarray>
Coordinates:
  * country  (country) <U32 26kB 'Armenia' 'Azerbaijan' ... 'Togo'
    region   (country) <U28 23kB 'Central Asia' ... 'Western Sub-Saharan Africa'
  * year     (year) int64 272B 2035 2036 2037 2038 2039 ... 2065 2066 2067 2068
  * sample   (sample) int64 8kB 0 1 2 3 4 5 6 7 ... 993 994 995 996 997 998 999